### **Instalación**

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-mistralai faiss-cpu sentence-transformers mistralai

### **Imports**

In [ ]:
import os
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import create_retriever_tool
from langchain import hub

print("Librerias importadas.")

### **Configurar API Key**

In [10]:
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
print("API Key cargada.")

API Key cargada.


### **Cargar dataset normalizado**

In [11]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("ventas_normalizado.csv")
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(3)

Saving ventas_normalizado.csv to ventas_normalizado (1).csv
Dataset cargado: 2823 filas, 26 columnas


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,PRODUCTLINE,MSRP,PRODUCTCODE,...,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE,YEAR,MONTH,QUARTER,DAY,DEALSIZE_CODE,STATUS_CODE
0,10107,30,95.70,2,2871.00,2003-02-24,SHIPPED,MOTORCYCLES,95,S10_1678,...,NaN,Yu,Kwai,SMALL,2003,2,1,24,1,1
1,10121,34,81.35,5,2765.90,2003-05-07,SHIPPED,MOTORCYCLES,95,S10_1678,...,EMEA,Henriot,Paul,SMALL,2003,5,2,7,1,1
2,10134,41,94.74,2,3884.34,2003-07-01,SHIPPED,MOTORCYCLES,95,S10_1678,...,EMEA,Da Cunha,Daniel,MEDIUM,2003,7,3,1,2,1


### **Convertir DataFrame a Documents**

In [12]:
documents = []
for _, row in df.iterrows():
    texto = (
        f"Orden: {row['ORDERNUMBER']} | "
        f"Cliente: {row['CUSTOMERNAME']} | "
        f"Pais: {row['COUNTRY']} | "
        f"Producto: {row['PRODUCTLINE']} | "
        f"Ventas: {row['SALES']} | "
        f"Estado: {row['STATUS']} | "
        f"Fecha: {row['ORDERDATE']} | "
        f"Trimestre: {row['QUARTER']} | "
        f"Anio: {row['YEAR']} | "
        f"Tamano deal: {row['DEALSIZE']}"
    )
    documents.append(Document(
        page_content=texto,
        metadata={
            "pais": str(row["COUNTRY"]),
            "producto": str(row["PRODUCTLINE"]),
            "anio": str(row["YEAR"])
        }
    ))
print(f"Documentos creados: {len(documents)}")

Documentos creados: 2823


### **Chunking**

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Chunks generados: {len(chunks)}")

Chunks generados: 2823


### **Embeddings y FAISS**

In [16]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS listo: {vectorstore.index.ntotal} vectores")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS listo: 2823 vectores


### **Retriever**

In [17]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
test = retriever.invoke("pais con mas ventas")
print(f"Retriever OK - docs recuperados: {len(test)}")
print(test[0].page_content[:150])

Retriever OK - docs recuperados: 5
Orden: 10213 | Cliente: DOUBLE DECKER GIFT STORES, LTD | Pais: UK | Producto: VINTAGE CARS | Ventas: 3602.02 | Estado: SHIPPED | Fecha: 2004-01-22 | T


### **Tool del agente**

In [18]:
retriever_tool = create_retriever_tool(
    retriever,
    name="consulta_ventas",
    description=(
        "Busca informacion del dataset de ventas comerciales. "
        "Usala para responder preguntas sobre paises, clientes, "
        "productos, fechas, montos y estados de ordenes."
    )
)
tools = [retriever_tool]
print("Tool creada.")

Tool creada.


### **RAG directo**

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
Eres un experto en analisis de ventas comerciales.
Responde usando SOLO la informacion del contexto.

CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:
""")

llm = ChatMistralAI(model="mistral-small-latest", temperature=0)

def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

cadena_rag = (
    {"context": retriever | formatear_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Cadena RAG lista.")


Cadena RAG lista.


### **Prueba**

In [23]:
respuesta = cadena_rag.invoke("Que pais tuvo mas ventas?")
print(respuesta)

UK


### **Chat interactivo**

In [24]:
print("AGENTE RAG - VENTAS COMERCIALES")
print("Escribi 'salir' para terminar.\n")
while True:
    pregunta = input("Pregunta: ")
    if pregunta.lower() == "salir":
        print("Chat finalizado.")
        break
    try:
        respuesta = cadena_rag.invoke(pregunta)
        print("\nAgente:", respuesta)
        print("-" * 40)
    except Exception as e:
        print("Error:", e)

AGENTE RAG - VENTAS COMERCIALES
Escribi 'salir' para terminar.


Agente: Los productos más vendidos son:

1. **CLASSIC CARS** (4 ventas totales)
2. **TRUCKS AND BUSES** (1 venta)
----------------------------------------

Agente: TECHNICS STORES INC.
----------------------------------------
Pregunta: salir
Chat finalizado.
